# Lab: Advanced AI Agent Concepts

This lab teaches five core ideas behind learning agents by having you **build the
key logic yourself**, not just run finished code. For each concept you get a working
harness (the setup, the examples, the visualization). The important part, the actual
learning rule, is left blank for you to write.

### How to use this notebook
- Cells marked `# TODO` are yours to complete. They will raise
  `NotImplementedError` until you finish them.
- Run cells top to bottom with **Shift + Enter**. Later parts reuse earlier code,
  so order matters.
- After each part there is an **Experiment** cell (change something and rerun) and a
  **Reflection** cell (answer in your own words, based on what you saw).
- Do not use a text AI to write your code or your answers. Your explanations must
  match the output your own code produced.


In [1]:
# Setup (ready to run)
import random
print("Ready.")

Ready.


## Part 1 — Deep Q-Network (DQN)

A DQN learns the **value** of taking an action in a state. Here we store those values
in a Q-table (a dictionary mapping `(state, action)` to a number). After each
experience, we nudge the stored value toward what we actually observed.

**The Q-learning update formula:**
```
new_value = old_value + learning_rate * (reward + discount_factor * next_max - old_value)
```
where `next_max` is the best Q-value available from the next state.

**Your task (Cell below):** implement the `update` method using that formula.


In [2]:
class SimpleDQN:
    def __init__(self):
        self.q_table = {}            # (state, action) -> value
        self.learning_rate = 0.1
        self.discount_factor = 0.9

    def get_q_value(self, state, action):
        # Ready to use. Returns 0.0 for unseen pairs.
        return self.q_table.get((state, action), 0.0)

    def update(self, state, action, reward, next_state):
        # TODO: implement the Q-learning update.
        # 1. old_value = current Q-value for (state, action)
        old_value = self.get_q_value(state, action)

        # 2. next_max  = the highest Q-value over actions ['move', 'stop'] in next_state
        next_max = max(
            self.get_q_value(next_state, "move"),
            self.get_q_value(next_state, "stop")
        )

        # 3. new_value = old_value + learning_rate * (reward + discount_factor * next_max - old_value)
        new_value = old_value + self.learning_rate * (
            reward + self.discount_factor * next_max - old_value
        )

        # 4. store new_value back into self.q_table[(state, action)]
        self.q_table[(state, action)] = new_value


In [3]:
# Run this to test your update (provided)
agent = SimpleDQN()
agent.update("obstacle_ahead", "stop", reward=10, next_state="clear_path")
print("After 1 update:", agent.get_q_value("obstacle_ahead", "stop"))

# Apply the SAME experience four more times and watch the value climb
for _ in range(4):
    agent.update("obstacle_ahead", "stop", reward=10, next_state="clear_path")
    print("Q-value:", round(agent.get_q_value("obstacle_ahead", "stop"), 4))

After 1 update: 1.0
Q-value: 1.9
Q-value: 2.71
Q-value: 3.439
Q-value: 4.0951


In [4]:
# EXPERIMENT (TODO)
# Make a new agent, set its learning_rate to 0.5, and repeat the same five updates.
# Compare how fast the value rises against the 0.1 agent above.

# TODO: write your experiment here.
agent_fast = SimpleDQN()
agent_fast.learning_rate = 0.5

for _ in range(5):
  agent_fast.update("obstacle_ahead", "stop", reward=10, next_state="clear_path")
  print("Q-value:", round(agent_fast.get_q_value("obstacle_ahead", "stop"), 4))


Q-value: 5.0
Q-value: 7.5
Q-value: 8.75
Q-value: 9.375
Q-value: 9.6875


**Reflection 1.** In two to three sentences, explain why the Q-value rose with each
repeat, and what changing the learning rate did to how fast it rose.

_The Q-value kept going up because the agent kept getting the same positive reward, so it learned that choosing "stop" was the right action. When I changed the learning rate from 0.1 to 0.5, the Q-value increased much faster because the agent learned more from each update._


## Part 2 — Policy Gradient

A policy-gradient agent does not store values. It stores a **probability** for each
action and shifts those probabilities toward actions that earned reward.

**Your task:** implement `update_policy` so that when an action earns positive reward,
its probability goes up, and then all probabilities are renormalized to sum to 1.


In [5]:
class SimplePolicyAgent:
    def __init__(self):
        self.action_probabilities = {'move': 0.5, 'stop': 0.5}
        self.learning_rate = 0.1

    def choose_action(self):
        # Ready to use. Samples an action using the current probabilities.
        return random.choices(
            list(self.action_probabilities.keys()),
            list(self.action_probabilities.values())
        )[0]

    def update_policy(self, action, reward):
        # TODO:
        # If reward > 0:
        #   1. add self.learning_rate to the probability of `action`
        #   2. renormalize: divide every probability by the new total so they sum t


        if reward > 0:
          self.action_probabilities[action] += self.learning_rate

          total = sum(self.action_probabilities.values())
          for a in self.action_probabilities:
            self.action_probabilities[a] /= total


In [6]:
# Run this to test (provided). Reward 'move', punish 'stop'.
policy_agent = SimplePolicyAgent()
for _ in range(8):
    a = policy_agent.choose_action()
    reward = 1 if a == 'move' else -1
    policy_agent.update_policy(a, reward)
    print({k: round(v, 3) for k, v in policy_agent.action_probabilities.items()})

{'move': 0.545, 'stop': 0.455}
{'move': 0.587, 'stop': 0.413}
{'move': 0.624, 'stop': 0.376}
{'move': 0.658, 'stop': 0.342}
{'move': 0.69, 'stop': 0.31}
{'move': 0.69, 'stop': 0.31}
{'move': 0.718, 'stop': 0.282}
{'move': 0.743, 'stop': 0.257}


In [7]:
# EXPERIMENT (TODO)
# Copy the loop above but FLIP the reward rule so 'stop' is rewarded instead.
# Run it and watch which way the probabilities drift.

# TODO: write your experiment here.
policy_agent = SimplePolicyAgent()

for _ in range(8):
  a = policy_agent.choose_action()
  reward = 1 if a == 'stop' else -1
  policy_agent.update_policy(a, reward)
  print({k: round(v, 3) for k, v in policy_agent.action_probabilities.items()})

{'move': 0.5, 'stop': 0.5}
{'move': 0.5, 'stop': 0.5}
{'move': 0.455, 'stop': 0.545}
{'move': 0.413, 'stop': 0.587}
{'move': 0.413, 'stop': 0.587}
{'move': 0.413, 'stop': 0.587}
{'move': 0.376, 'stop': 0.624}
{'move': 0.342, 'stop': 0.658}


**Reflection 2.** In two to three sentences, describe how the action probabilities
changed once you flipped the reward, and how this differs from how the DQN in Part 1
learned.

_When I flipped the reward, the probability of choosing "stop" kept increasing while the probability of "move" decreased. This is different from the DQN in Part 1 because the policy agent learned by changing the action probabilities directly, while the DQN learned by updating Q-values for each action based on the rewards it received._


## Part 3 — Multi-Agent System

Several agents share one world. Each can **sense**, **decide**, and **act**. They also
**communicate** (here, just by seeing where the others are) so they can avoid collisions.

The agent, the world, and the visualization are provided. **Your task** is the
coordination rule: an agent should not move forward if another agent is in the cell
directly ahead of it.


In [8]:
# Provided: a single agent, the world, and a text visualizer.
class SimpleAgent:
    def __init__(self, world_size):
        self.position = random.randint(0, world_size - 1)
        self.world_size = world_size

    def sense(self, world):
        return world[self.position]

    def decide(self, observation):
        return 'move' if observation == ' ' else 'stop'

    def act(self, action):
        if action == 'move' and self.position < self.world_size - 1:
            self.position += 1

def create_world(size):
    return [' ' for _ in range(size)]

def visualize(world, positions):
    cells = []
    for i in range(len(world)):
        if i in positions:
            cells.append(str(positions.index(i) + 1))
        else:
            cells.append(world[i])
    return "[" + "][".join(cells) + "]"

print("Helpers ready.")

Helpers ready.


In [9]:
class SimpleMultiAgentSystem:
    def __init__(self, num_agents=3, world_size=10):
        self.agents = [SimpleAgent(world_size) for _ in range(num_agents)]
        self.world_size = world_size
        self.world = create_world(world_size)

    def communicate(self, agent_index):
        # Ready to use. Returns the positions of all the OTHER agents.
        return [a.position for i, a in enumerate(self.agents) if i != agent_index]

    def coordinate_actions(self):
        for i, agent in enumerate(self.agents):
            other_positions = self.communicate(i)
            observation = agent.sense(self.world)

            # TODO: decide the action.
            # If another agent is directly ahead (at agent.position + 1), the action
            # should be 'stop'. Otherwise use agent.decide(observation).
            # Then call agent.act(action).
            if agent.position + 1 in other_positions:
              action = 'stop'
            else:
              action = agent.decide(observation)

            agent.act(action)


In [10]:
# Run this to test (provided)
system = SimpleMultiAgentSystem(num_agents=3)
for step in range(5):
    system.coordinate_actions()
    positions = [a.position for a in system.agents]
    print(f"Step {step+1}: {visualize(system.world, positions)}")

Step 1: [ ][1][ ][2][ ][3][ ][ ][ ][ ]
Step 2: [ ][ ][1][ ][2][ ][3][ ][ ][ ]
Step 3: [ ][ ][ ][1][ ][2][ ][3][ ][ ]
Step 4: [ ][ ][ ][ ][1][ ][2][ ][3][ ]
Step 5: [ ][ ][ ][ ][ ][1][ ][2][ ][3]


In [11]:
# EXPERIMENT (TODO)
# Run the same simulation again with num_agents = 5 and watch how often agents stop.

# TODO: write your experiment here.
system_five = SimpleMultiAgentSystem(num_agents=5)

for step in range(5):
  system_five.coordinate_actions()
  positions = [a.position for a in system_five.agents]
  print(f"Step {step+1}: {visualize(system_five.world, positions)}")

Step 1: [ ][ ][ ][ ][4][3][ ][ ][2][1]
Step 2: [ ][ ][ ][ ][ ][4][3][ ][2][1]
Step 3: [ ][ ][ ][ ][ ][ ][4][3][2][1]
Step 4: [ ][ ][ ][ ][ ][ ][4][3][2][1]
Step 5: [ ][ ][ ][ ][ ][ ][4][3][2][1]


**Reflection 3.** In two to three sentences, describe what the coordination rule does
when agents get close, and what changed when you used five agents instead of three.

_The coordination rule made the agents stop whenever another agent was directly in front of them, which helped prevent collisions. When I increased the number of agents from 3 to 5, the agents had to stop more often because there were more agents sharing the same space, making it more likely that one would be directly ahead of another._


## Part 4 — Federated Learning

Several agents each learn on their own, then **share knowledge by averaging** their
Q-tables, without ever sharing raw experience. After aggregation, every agent holds the
same averaged values.

**Your task:** implement `aggregate_knowledge` to average the Q-values across all agents
and write the averaged table back into each one. (Reuses `SimpleDQN` from Part 1.)


In [12]:
class SimpleFederatedSystem:
    def __init__(self, num_agents=3):
        self.agents = [SimpleDQN() for _ in range(num_agents)]

    def aggregate_knowledge(self):
        # TODO:
        # 1. Collect every (state, action) key that appears in ANY agent's q_table.
        # 2. For each key, average that value across all agents (use 0.0 if an agent
        #    has not seen it).
        # 3. Replace every agent's q_table with a copy of the averaged table.

        all_keys = set()

        for agent in self.agents:
          all_keys.update(agent.q_table.keys())

        averaged_table = {}

        for key in all_keys:
          total = sum(agent.q_table.get(key, 0.0) for agent in self.agents)
          averaged_table[key] = total / len(self.agents)

        for agent in self.agents:
          agent.q_table = averaged_table.copy()


In [13]:
# Run this to test (provided)
fed = SimpleFederatedSystem(num_agents=3)
for i, agent in enumerate(fed.agents):
    agent.q_table[(f"position_{i}", "move")] = round(random.random(), 3)

print("Before aggregation:")
for i, agent in enumerate(fed.agents):
    print(f"  Agent {i}: {agent.q_table}")

fed.aggregate_knowledge()

print("\nAfter aggregation:")
for i, agent in enumerate(fed.agents):
    print(f"  Agent {i}: {agent.q_table}")

Before aggregation:
  Agent 0: {('position_0', 'move'): 0.946}
  Agent 1: {('position_1', 'move'): 0.834}
  Agent 2: {('position_2', 'move'): 0.338}

After aggregation:
  Agent 0: {('position_2', 'move'): 0.11266666666666668, ('position_1', 'move'): 0.27799999999999997, ('position_0', 'move'): 0.3153333333333333}
  Agent 1: {('position_2', 'move'): 0.11266666666666668, ('position_1', 'move'): 0.27799999999999997, ('position_0', 'move'): 0.3153333333333333}
  Agent 2: {('position_2', 'move'): 0.11266666666666668, ('position_1', 'move'): 0.27799999999999997, ('position_0', 'move'): 0.3153333333333333}


**Reflection 4.** In two to three sentences, explain what aggregation did to each
agent's values and give one real situation where sharing averaged knowledge, instead of
raw data, would be useful.

_Aggregation combined the Q-values from all three agents and gave each agent the same averaged table. This could be useful in hospitals, where different hospitals can improve a shared AI model without sending patients' private medical records to each other._


## Part 5 — Compare Learning Approaches

Finally, run both kinds of agent over many episodes and compare their average reward.
The loop is provided, but **you must call the function** at the bottom, since the
original code defined it and never ran it.


In [14]:
def compare_learning_approaches(episodes=100):
    dqn_agent = SimpleDQN()
    policy_agent = SimplePolicyAgent()
    dqn_rewards, policy_rewards = [], []

    for episode in range(episodes):
        # DQN: pick the action with the highest current Q-value
        state, dqn_total = "start", 0
        for _ in range(5):
            action = max(['move', 'stop'], key=lambda a: dqn_agent.get_q_value(state, a))
            reward = random.choice([-1, 1])
            next_state = f"state_{random.randint(1, 5)}"
            dqn_agent.update(state, action, reward, next_state)
            dqn_total += reward
            state = next_state

        # Policy gradient
        policy_total = 0
        for _ in range(5):
            action = policy_agent.choose_action()
            reward = random.choice([-1, 1])
            policy_agent.update_policy(action, reward)
            policy_total += reward

        dqn_rewards.append(dqn_total)
        policy_rewards.append(policy_total)

        if episode % 10 == 0:
            print(f"Episode {episode:>3} | "
                  f"DQN avg {sum(dqn_rewards[-10:]) / 10:+.1f} | "
                  f"Policy avg {sum(policy_rewards[-10:]) / 10:+.1f}")

    return dqn_rewards, policy_rewards

In [15]:
# TODO: call compare_learning_approaches() and run it.
dqn_rewards, policy_rewards = compare_learning_approaches()

Episode   0 | DQN avg -0.1 | Policy avg +0.1
Episode  10 | DQN avg -0.4 | Policy avg +0.6
Episode  20 | DQN avg +0.0 | Policy avg -1.0
Episode  30 | DQN avg +0.2 | Policy avg +0.0
Episode  40 | DQN avg +0.8 | Policy avg -0.4
Episode  50 | DQN avg +1.6 | Policy avg +0.2
Episode  60 | DQN avg -0.6 | Policy avg -1.2
Episode  70 | DQN avg -0.2 | Policy avg -0.2
Episode  80 | DQN avg +0.8 | Policy avg -0.8
Episode  90 | DQN avg +0.4 | Policy avg +0.8


**Reflection 5.** In two to three sentences, state which approach showed steadier
average reward in your run and offer one reason why. (The reward here is random, so look
at the trend, not a single number.)

_In my run, the DQN showed a slightly steadier average reward because it had more positive results over time, even though both approaches went up and down. I think this happened because the DQN learns by updating its Q-values from previous rewards, while the policy agent adjusts its action probabilities directly, making it more affected by the random rewards._


## Final Reflection and Submission

**Final reflection (150 to 250 words).** Which of the five concepts was clearest to you,
which was hardest, and what is one real situation where a multi-agent or federated
approach would beat a single agent? Reference specific output you saw.

_The concept that was clearest to me was the Deep Q-Network (DQN). It was easy to see how the Q-value increased every time the agent received the same positive reward. In my experiment, the value kept rising, and when I changed the learning rate from 0.1 to 0.5, it increased much faster. The hardest concept for me was federated learning because I had to understand how several agents could combine their knowledge by averaging their Q-values without sharing their original experiences. After running the aggregation, I saw that every agent ended up with the same averaged Q-table, even though each one started with different values. I also thought the multi-agent system was interesting because the agents stopped moving whenever another agent was directly in front of them, which prevented collisions. When I increased the number of agents from three to five, they had to stop more often because there was less free space. A real situation where a multi-agent or federated approach would be better than a single agent is in self-driving cars or hospitals. Multiple cars can share information about traffic, or different hospitals can improve a shared AI model while keeping patient records private. This lab helped me understand how different AI learning approaches solve problems in different ways._

---

### Before you submit
- [ ] Every `# TODO` is implemented and no cell raises `NotImplementedError`.
- [ ] All cells run top to bottom with outputs visible.
- [ ] Both experiment cells (Parts 1, 2, 3, 5) contain your changes and their output.
- [ ] All six reflections are answered in your own words.
- [ ] Export to PDF with **File > Print > Save as PDF** and submit on Canvas.
